In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('../../data/gold_policy_clean.csv', parse_dates=['date'])

DUTY_CUT    = pd.Timestamp('2024-07-24')
POLICY_DATE = pd.Timestamp('2026-05-13')
ITS_BETA1   = 9743.0   # headline from Notebook 03

# ── Train / test split ─────────────────────────────────────────────────────
cols = ['date','domestic_premium','delta_Gold_USD','delta_FX']

train = df[(df['date'] >= DUTY_CUT) & (df['date'] < POLICY_DATE)][cols].dropna().reset_index(drop=True)
test  = df[df['date'] >= POLICY_DATE][cols].dropna().reset_index(drop=True)

print(f'TRAIN (pre-hike, low-duty window):')
print(f'  N={len(train)}, {train["date"].iloc[0].date()} → {train["date"].iloc[-1].date()}')
print(f'  Premium mean: ₹{train["domestic_premium"].mean():,.1f}')
print()
print(f'TEST (post-hike):')
print(f'  N={len(test)}, {test["date"].iloc[0].date()} → {test["date"].iloc[-1].date()}')
print(f'  Premium mean: ₹{test["domestic_premium"].mean():,.1f}')
print()
print(f'ITS β₁ to beat: ₹{ITS_BETA1:,.0f}')

TRAIN (pre-hike, low-duty window):
  N=346, 2024-07-25 → 2026-05-12
  Premium mean: ₹-54.5

TEST (post-hike):
  N=22, 2026-05-13 → 2026-06-16
  Premium mean: ₹10,156.1

ITS β₁ to beat: ₹9,743


In [3]:
# Cell 2 — Select ARIMA order (without exogenous — pmdarima issue)
# We use auto_arima just for order selection, then fit SARIMAX with exogenous separately
from pmdarima import auto_arima

y_train    = train['domestic_premium']
exog_train = train[['delta_Gold_USD','delta_FX']]
exog_test  = test[['delta_Gold_USD','delta_FX']]

order_sel = auto_arima(
    y_train,
    seasonal=False,
    information_criterion='aic',
    max_p=5, max_q=5, max_d=1,
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore'
)

p, d, q = order_sel.order
print(f'Selected order: ARIMA({p},{d},{q})')
print(f'AIC (without exog): {order_sel.aic():.2f}')
print()
print(f'd={d} → series {"stationary, no differencing needed" if d==0 else "needed differencing"}')
print(f'Confirms ADF result from TEST 02')

Selected order: ARIMA(1,0,1)
AIC (without exog): 6029.56

d=0 → series stationary, no differencing needed
Confirms ADF result from TEST 02
